# US Stock Quant Strategy — S&P 500

**목표:** 한국 주식 전략을 미국 S&P 500에 적용한다

**순서:**
1. 라이브러리 설치 및 임포트
2. S&P 500 종목 리스트 수집
3. 주가 데이터 수집
4. 재무 데이터 수집 (SimFin)
5. 팩터 계산 (가치, 모멘텀, 퀄리티)
6. XGBoost 신호 생성
7. 백테스팅
8. 성과 분석

---
## Cell 1 — 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import xgboost as xgb
import shap
import warnings
import os
import requests

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# 폴더 생성
for folder in ['data/raw', 'data/processed',
               'data/factors', 'results']:
    os.makedirs(folder, exist_ok=True)

# ── SimFin API Key 설정 ───────────────────────────────────
# simfin.com 가입 후 API Key 입력
SIMFIN_API_KEY = 'YOUR_API_KEY_HERE'  # ← 여기에 입력

print('라이브러리 임포트 완료')
print(f'pandas  : {pd.__version__}')
print(f'yfinance: {yf.__version__}')
print(f'xgboost : {xgb.__version__}')

---
## Cell 2 — S&P 500 종목 리스트 수집

> **한국 전략과 차이점:**  
> 한국은 FinanceDataReader로 종목 수집  
> 미국은 Wikipedia에서 S&P 500 리스트 수집 (무료, 항상 최신)

In [ ]:
def get_sp500_tickers():
    """
    S&P 500 구성 종목 수집
    Wikipedia에서 실시간으로 가져옴
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url)
    sp500  = tables[0]

    # 티커 정리 (BRK.B → BRK-B, yfinance 형식)
    tickers = sp500['Symbol'].str.replace('.', '-', regex=False).tolist()
    names   = dict(zip(tickers, sp500['Security'].tolist()))
    sectors = dict(zip(tickers, sp500['GICS Sector'].tolist()))

    print(f'S&P 500 종목 수집 완료: {len(tickers)}개')

    # 섹터별 분포
    sector_counts = sp500['GICS Sector'].value_counts()
    print('\n섹터별 종목 수:')
    for sector, cnt in sector_counts.items():
        print(f'  {sector:35s}: {cnt}개')

    return tickers, names, sectors


tickers, names, sectors = get_sp500_tickers()

# 실습용: 상위 100개만 사용 (속도)
# 실전에서는 전체 500개 사용
tickers_100 = tickers[:100]
print(f'\n실습용 종목: {len(tickers_100)}개 (전체 {len(tickers)}개 중)')

---
## Cell 3 — 주가 데이터 수집

> **미국 주식 특징:**  
> 거래일이 한국과 다름 (미국 공휴일 제외)  
> 배당이 한국보다 많아서 수정 종가 더 중요  
> 환율 영향 없음 (달러 기준)

In [ ]:
START_DATE = '2018-01-01'
END_DATE   = '2024-01-01'

print(f'주가 수집 중... ({len(tickers_100)}개 종목, {START_DATE}~{END_DATE})')
print('약 1~2분 소요')

raw = yf.download(
    tickers_100,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True
)

prices = raw['Close'].copy()
prices.columns = prices.columns.astype(str)

# 데이터 70% 이상 있는 종목만 유지
valid = prices.columns[prices.notna().mean() >= 0.7]
prices = prices[valid]

# 로그 수익률
log_returns = np.log(prices / prices.shift(1)).clip(-0.5, 0.5)

print(f'\n수집 완료')
print(f'  유효 종목 수  : {len(valid)}개')
print(f'  거래일 수     : {len(prices)}일')
print(f'  기간          : {prices.index[0].date()} ~ {prices.index[-1].date()}')

# 저장
prices.to_csv('data/raw/us_prices.csv')
log_returns.to_csv('data/processed/us_log_returns.csv')
print('저장 완료')

In [ ]:
# 대표 종목 시각화
sample = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
sample = [t for t in sample if t in prices.columns]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for ax, ticker, color in zip(axes.flatten(), sample, colors):
    norm = prices[ticker].dropna() / prices[ticker].dropna().iloc[0] * 100
    norm.plot(ax=ax, color=color, lw=1.5)
    ax.set_title(f'{ticker} — {names.get(ticker, ticker)}')
    ax.set_ylabel('Normalized Price (start=100)')
    ax.axhline(100, color='gray', lw=0.8, linestyle='--')

plt.suptitle('S&P 500 Representative Stocks (2018~2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/us_01_price_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/us_01_price_trends.png')

---
## Cell 4 — 재무 데이터 수집 (SimFin)

> **한국 전략과 차이점:**  
> 한국: pykrx로 PER, PBR 수집  
> 미국: SimFin API로 실제 재무제표 수집  
> SimFin은 PER, PBR뿐 아니라 ROE, 영업이익률까지 제공  
> → 더 정확한 가치/퀄리티 팩터 계산 가능

In [ ]:
def get_simfin_data(api_key, tickers, start='2018-01-01'):
    """
    SimFin API로 미국 주식 재무 데이터 수집

    수집 항목:
    - Price-to-Book (PBR)
    - Price-to-Earnings (PER)
    - Return on Equity (ROE)
    - Profit Margin (영업이익률)
    """
    base_url = 'https://backend.simfin.com/api/v3'
    headers  = {'Authorization': f'api-key {api_key}'}

    all_data = []
    # 티커 20개씩 배치 처리
    batch_size = 20

    print(f'SimFin 재무 데이터 수집 중...')
    print(f'총 {len(tickers)}개 종목, {batch_size}개씩 배치 처리')

    for i in range(0, min(len(tickers), 100), batch_size):
        batch = tickers[i:i+batch_size]
        ticker_str = ','.join(batch)

        try:
            # 주가 기반 비율 (PER, PBR 등)
            url = f'{base_url}/companies/prices/ratios'
            params = {
                'ticker' : ticker_str,
                'start'  : start,
                'end'    : '2024-01-01',
                'period' : 'quarterly'
            }
            resp = requests.get(url, headers=headers, params=params)

            if resp.status_code == 200:
                data = resp.json()
                if data:
                    df = pd.DataFrame(data)
                    all_data.append(df)
                    print(f'  배치 {i//batch_size + 1}: {len(batch)}개 종목 완료')
            else:
                print(f'  배치 {i//batch_size + 1}: 오류 {resp.status_code}')

        except Exception as e:
            print(f'  배치 {i//batch_size + 1}: 실패 ({e})')
            continue

    if not all_data:
        print('SimFin 데이터 없음 → 주가 기반 팩터만 사용')
        return None

    financials = pd.concat(all_data, ignore_index=True)
    print(f'\n수집 완료: {financials.shape}')
    print(f'컬럼: {list(financials.columns)}')

    return financials


# API Key 입력했으면 실행
if SIMFIN_API_KEY != 'YOUR_API_KEY_HERE':
    financials_us = get_simfin_data(
        SIMFIN_API_KEY, tickers_100
    )
    if financials_us is not None:
        financials_us.to_csv('data/raw/us_financials.csv', index=False)
        print('저장: data/raw/us_financials.csv')
else:
    print('API Key를 Cell 1에 입력하세요')
    print('simfin.com → 로그인 → Account → API Key')
    print('\n일단 주가 기반 팩터로 진행합니다 (Cell 5로 이동)')
    financials_us = None

---
## Cell 5 — 팩터 계산

> **한국 전략에서 배운 것 적용:**  
> 모멘텀 IC = 음수 → 반전 필요했음  
> 미국에서는 어떻게 나오는지 다시 확인  
> 미국은 모멘텀 효과가 한국보다 강하게 나타남

In [ ]:
def cross_sectional_zscore(df):
    """횡단면 Z-score 표준화"""
    mean = df.mean(axis=1)
    std  = df.std(axis=1)
    z    = df.sub(mean, axis=0).div(std, axis=0)
    return z.clip(-3, 3)


print('팩터 계산 중...')

# ── 모멘텀 팩터 ───────────────────────────────────────────
# 12개월 수익률 - 최근 1개월 (단기 반전 방지)
mom_12m  = log_returns.rolling(252).sum()
mom_1m   = log_returns.rolling(21).sum()
mom_raw  = mom_12m - mom_1m
mom_z    = cross_sectional_zscore(mom_raw)
print('  모멘텀 팩터 완료')

# ── 가치 팩터 ─────────────────────────────────────────────
# SimFin 재무 데이터가 있으면 PBR 사용
# 없으면 주가 기반 대용변수
if financials_us is not None and 'pb' in financials_us.columns:
    # 실제 PBR 사용 (낮을수록 가치주)
    print('  가치 팩터: 실제 PBR 사용')
    # PBR 피벗 테이블 생성
    pb_pivot = financials_us.pivot_table(
        index='date', columns='ticker', values='pb'
    )
    pb_pivot.index = pd.to_datetime(pb_pivot.index)
    pb_daily = pb_pivot.reindex(prices.index, method='ffill')
    value_z  = cross_sectional_zscore(-pb_daily)  # 낮은 PBR = 높은 가치
else:
    # 주가 기반 대용변수
    print('  가치 팩터: 주가 기반 대용변수 사용')
    high_52w  = prices.rolling(252).max()
    value_raw = -(prices / high_52w)  # 낮을수록 가치주
    # 장기 반전 팩터와 결합
    reversal  = -log_returns.rolling(252*3).sum()
    value_z   = cross_sectional_zscore(
        (cross_sectional_zscore(value_raw) +
         cross_sectional_zscore(reversal)) / 2
    )
print('  가치 팩터 완료')

# ── 퀄리티 팩터 ──────────────────────────────────────────
# 저변동성 + 수익 일관성 + 롤링 샤프
vol_63        = log_returns.rolling(63).std() * np.sqrt(252)
pos_ratio     = log_returns.rolling(252).apply(
    lambda x: (x > 0).mean(), raw=True
)
roll_sharpe   = (
    log_returns.rolling(252).mean() * 252 /
    (log_returns.rolling(252).std() * np.sqrt(252))
)
quality_raw   = (
    cross_sectional_zscore(-vol_63) +
    cross_sectional_zscore(pos_ratio) +
    cross_sectional_zscore(roll_sharpe)
) / 3
quality_z     = cross_sectional_zscore(quality_raw)
print('  퀄리티 팩터 완료')

print(f'\n팩터 Shape: {mom_z.shape}')

In [ ]:
# ── IC 부호 점검 ──────────────────────────────────────────
# 한국에서는 모멘텀/퀄리티가 반전 필요했음
# 미국에서는 어떤지 확인

fwd_21 = log_returns.rolling(21).sum().shift(-21)
monthly_dates = log_returns.resample('BM').last().index
monthly_dates = monthly_dates[
    (monthly_dates >= mom_z.first_valid_index()) &
    (monthly_dates <= log_returns.index[-25])
]

print('미국 팩터 IC 부호 점검:')
print(f'  {"팩터":12s} | {"평균 IC":>8} | 방향')
print('  ' + '-' * 35)

ic_signs = {}
for name, factor in [('모멘텀', mom_z),
                      ('가치',   value_z),
                      ('퀄리티', quality_z)]:
    ic_list = []
    for date in monthly_dates:
        if date not in factor.index or date not in fwd_21.index:
            continue
        f = factor.loc[date].dropna()
        r = fwd_21.loc[date].dropna()
        common = f.index.intersection(r.index)
        if len(common) < 10:
            continue
        ic, _ = stats.spearmanr(f[common], r[common])
        ic_list.append(ic)
    mean_ic = np.mean(ic_list) if ic_list else 0
    direction = '정방향' if mean_ic > 0 else '반전 필요'
    mark = '✅' if mean_ic > 0 else '❌'
    ic_signs[name] = mean_ic
    print(f'  {name:12s} | {mean_ic:>8.4f} | {mark} {direction}')

# 반전 필요한 팩터 자동 처리
print('\n부호 수정 적용:')
if ic_signs.get('모멘텀', 0) < 0:
    mom_z = -mom_z
    print('  모멘텀: 반전 적용')
else:
    print('  모멘텀: 그대로')

if ic_signs.get('가치', 0) < 0:
    value_z = -value_z
    print('  가치  : 반전 적용')
else:
    print('  가치  : 그대로')

if ic_signs.get('퀄리티', 0) < 0:
    quality_z = -quality_z
    print('  퀄리티: 반전 적용')
else:
    print('  퀄리티: 그대로')

# 팩터 저장
mom_z.to_csv('data/factors/us_momentum.csv')
value_z.to_csv('data/factors/us_value.csv')
quality_z.to_csv('data/factors/us_quality.csv')
print('\n팩터 저장 완료')

---
## Cell 6 — XGBoost 신호 생성

In [ ]:
# 피처 매트릭스 구성
print('피처 매트릭스 구성 중...')

fwd_ret  = log_returns.rolling(21).sum().shift(-21)
vol_63d  = log_returns.rolling(63).std() * np.sqrt(252)

records = []
eval_dates = monthly_dates

for date in eval_dates:
    if date not in log_returns.index:
        diffs = abs(log_returns.index - date)
        date  = log_returns.index[diffs.argmin()]

    for ticker in prices.columns:
        try:
            f_mom  = mom_z.loc[date, ticker]     if date in mom_z.index     else np.nan
            f_val  = value_z.loc[date, ticker]   if date in value_z.index   else np.nan
            f_qual = quality_z.loc[date, ticker] if date in quality_z.index else np.nan
            f_1m   = mom_1m.loc[date, ticker]    if date in mom_1m.index    else np.nan
            f_vol  = vol_63d.loc[date, ticker]   if date in vol_63d.index   else np.nan
            target = fwd_ret.loc[date, ticker]   if date in fwd_ret.index   else np.nan

            if any(np.isnan([f_mom, f_val, f_qual, target])):
                continue

            records.append({
                'date'   : date, 'ticker': ticker,
                'mom'    : f_mom, 'value': f_val,
                'quality': f_qual,
                'mom_1m' : f_1m  if not np.isnan(f_1m)  else 0,
                'vol'    : f_vol if not np.isnan(f_vol) else 0.2,
                'target' : target
            })
        except:
            continue

feature_df = pd.DataFrame(records)
print(f'피처 매트릭스: {feature_df.shape}')
print(f'기간: {feature_df["date"].min().date()} ~ {feature_df["date"].max().date()}')

In [ ]:
# XGBoost 학습 (전체 데이터)
FEATURE_COLS = ['mom', 'value', 'quality', 'mom_1m', 'vol']

clean_df = feature_df.dropna(subset=FEATURE_COLS + ['target'])
sort_idx  = np.argsort(clean_df['date'].values)
clean_df  = clean_df.iloc[sort_idx].reset_index(drop=True)

X = clean_df[FEATURE_COLS].values
y = clean_df['target'].values

XGB_PARAMS = {
    'n_estimators'     : 300,
    'max_depth'        : 3,
    'learning_rate'    : 0.05,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_weight' : 20,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 1.0,
    'random_state'     : 42,
    'tree_method'      : 'hist',
    'verbosity'        : 0
}

print('XGBoost 학습 중...')
model = xgb.XGBRegressor(**XGB_PARAMS)
model.fit(X, y)
clean_df['prediction'] = model.predict(X)

# 전체 IC 확인
ic_full, _ = stats.spearmanr(
    clean_df['prediction'], clean_df['target']
)
print(f'학습 완료 — 전체 IC: {ic_full:.4f}')

# 포지션 생성 (롱온리, 상위 20종목)
all_pos = []
for date, group in clean_df.groupby('date'):
    if len(group) < 20:
        continue
    top20 = group.nlargest(20, 'prediction')['ticker'].tolist()
    for t in top20:
        all_pos.append({'date': date, 'ticker': t, 'weight': 1/20})

positions = pd.DataFrame(all_pos).pivot_table(
    index='date', columns='ticker', values='weight', fill_value=0
)
print(f'포지션 생성 완료: {positions.shape[0]}개월')

---
## Cell 7 — 백테스팅

In [ ]:
# ── 백테스팅 실행 ─────────────────────────────────────────
# 미국 거래 비용: 한국보다 낮음
# IBKR 기준 주당 $0.005, 슬리피지 포함 약 0.1%
TRANSACTION_COST = 0.001  # 0.1%

common_tickers = positions.columns.intersection(log_returns.columns)
positions      = positions[common_tickers]
lr_bt          = log_returns[common_tickers]

pos_daily = positions.reindex(lr_bt.index, method='ffill').fillna(0)
pos_lag   = pos_daily.shift(1).fillna(0)

gross_ret = (pos_lag * lr_bt).sum(axis=1)
turnover  = pos_lag.diff().abs().sum(axis=1)
costs     = turnover * TRANSACTION_COST
net_ret   = gross_ret - costs
net_ret   = net_ret[net_ret != 0]

# 벤치마크: SPY (S&P 500 ETF)
spy = yf.download('SPY', start=START_DATE,
                   end=END_DATE, auto_adjust=True,
                   progress=False)['Close']
spy_ret = np.log(spy / spy.shift(1)).dropna()
spy_ret = spy_ret.reindex(net_ret.index)

# 성과 지표
def perf(returns, label):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol
    cum     = (1 + returns).cumprod()
    mdd     = ((cum - cum.cummax()) / cum.cummax()).min()
    print(f'{label}:')
    print(f'  연수익률: {ann_ret:.2%}')
    print(f'  Sharpe  : {sharpe:.3f}')
    print(f'  MDD     : {mdd:.2%}')
    return ann_ret, sharpe, mdd

print('=' * 45)
print(' 백테스팅 성과 (2018~2024)')
print('=' * 45)
s_ret, s_sh, s_mdd = perf(net_ret, '전략 (롱온리 20종목)')
print()
b_ret, b_sh, b_mdd = perf(spy_ret.dropna(), 'SPY 벤치마크')
print('=' * 45)

beat = '✅ 벤치마크 초과' if s_sh > b_sh else '⚠️  벤치마크 미달'
print(f'\n결과: {beat}')

---
## Cell 8 — 성과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

cum_strategy = (1 + net_ret).cumprod()
cum_spy      = (1 + spy_ret.dropna()).cumprod()

# 누적 수익률
ax = axes[0, 0]
cum_strategy.plot(ax=ax, color='#2196F3', lw=2, label='Strategy')
cum_spy.plot(ax=ax, color='#9E9E9E', lw=1.5,
             linestyle='--', label='SPY (S&P 500)')
ax.set_title('Cumulative Return vs S&P 500')
ax.set_ylabel('Cumulative Return')
ax.legend()
ax.axhline(1, color='black', lw=0.5)

# 드로우다운
ax = axes[0, 1]
dd = (cum_strategy - cum_strategy.cummax()) / cum_strategy.cummax()
dd.plot(ax=ax, color='#EF5350', lw=1)
ax.fill_between(dd.index, dd, 0, color='#EF5350', alpha=0.3)
ax.axhline(s_mdd, color='darkred', lw=1.5, linestyle='--',
           label=f'MDD: {s_mdd:.2%}')
ax.set_title('Drawdown')
ax.set_ylabel('Drawdown')
ax.legend()

# 롤링 샤프
ax = axes[1, 0]
roll_sh = (
    net_ret.rolling(252).mean() * 252 /
    (net_ret.rolling(252).std() * np.sqrt(252))
)
roll_sh.plot(ax=ax, color='#9C27B0', lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1, color='green', lw=1, linestyle='--',
           alpha=0.7, label='Sharpe=1')
ax.fill_between(roll_sh.index, roll_sh, 0,
                where=roll_sh >= 0,
                color='#4CAF50', alpha=0.2)
ax.fill_between(roll_sh.index, roll_sh, 0,
                where=roll_sh < 0,
                color='#EF5350', alpha=0.2)
ax.set_title('Rolling 1-Year Sharpe Ratio')
ax.legend()

# 월별 수익률 히트맵
ax = axes[1, 1]
monthly = net_ret.resample('M').sum()
mdf = pd.DataFrame({
    'year' : monthly.index.year,
    'month': monthly.index.month,
    'ret'  : monthly.values
})
pivot = mdf.pivot(index='year', columns='month', values='ret')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                  'Jul','Aug','Sep','Oct','Nov','Dec']
sns.heatmap(pivot * 100, cmap='RdYlGn', center=0,
            annot=True, fmt='.1f', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Return (%)'})
ax.set_title('Monthly Returns Heatmap (%)')
ax.set_xlabel('')

plt.suptitle('US Strategy Performance (S&P 500 Universe)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/us_02_performance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/us_02_performance.png')

---
## Cell 9 — 연도별 분석 & 최종 리포트

In [ ]:
print('=' * 50)
print(' US Strategy — Final Report')
print('=' * 50)

print(f'\n기간: {net_ret.index[0].date()} ~ {net_ret.index[-1].date()}')
print(f'유니버스: S&P 500 상위 {len(common_tickers)}개 종목')
print(f'전략: 팩터 상위 20종목 롱온리, 월별 리밸런싱')

print(f'\n연도별 수익률:')
print(f'  {"Year":>6} | {"Strategy":>10} | {"SPY":>10} | 결과')
print('  ' + '-' * 38)

yearly_s   = net_ret.resample('Y').sum()
yearly_spy = spy_ret.dropna().resample('Y').sum()

for year in yearly_s.index:
    s = yearly_s.loc[year]
    b = yearly_spy.get(year, 0)
    mark = '✅' if s > b else '❌'
    print(f'  {year.year:>6} | {s:>10.2%} | {b:>10.2%} | {mark}')

beat_years = sum(
    yearly_s.loc[y] > yearly_spy.get(y, 0)
    for y in yearly_s.index
    if y in yearly_spy.index
)
total_years = len(yearly_s)
print(f'\n벤치마크 초과 연도: {beat_years}/{total_years}년')

# 모델 저장
import pickle
os.makedirs('data/models', exist_ok=True)
with open('data/models/us_xgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)

net_ret.to_csv('data/processed/us_net_returns.csv')
positions.to_csv('data/processed/us_positions.csv')

print('\n저장된 파일:')
for f in ['data/models/us_xgb_model.pkl',
          'data/processed/us_net_returns.csv',
          'data/processed/us_positions.csv',
          'results/us_01_price_trends.png',
          'results/us_02_performance.png']:
    print(f'  {f}')

print('\n' + '=' * 50)
print(' 완료! 다음: IBKR 연결 & 페이퍼 트레이딩')
print('=' * 50)